In [1]:
import numpy as np
from scipy.stats import norm
from sklearn.cluster import KMeans
import os
import pandas as pd
from sklearn.mixture import GaussianMixture
from tqdm import tqdm
import sys
sys.path.append('../utilities')
import plotting_utils as plotting
import simulation_utils as sim
import utilities as utils
import random
from scipy.stats import multivariate_normal
import math

In [2]:
class AdaptiveSampling:
    def __init__(self, target_dist, initial_proposal, n_iterations, n_samples_per_iter, condition):
        """
        Initialize the AdaptiveSampling object.

        Parameters:
        - target_dist: function, computes the target distribution density.
        - initial_proposal: list of dicts, parameters for the initial proposal distribution:
            [{'mean': ndarray, 'std': ndarray, 'weight': float}, ...].
        - n_iterations: int, number of adaptation iterations.
        - n_samples_per_iter: int, number of samples to draw per iteration.
        - condition: function, evaluates whether a sample satisfies the constraint.
        """
        self.target_dist = target_dist
        self.proposals = initial_proposal
        self.n_iterations = n_iterations
        self.n_samples_per_iter = n_samples_per_iter
        self.all_samples = []
        self.all_weights = []
        self.condition = condition

    def sample_from_proposal(self):
        """Generate samples from the current proposal mixture with conditioned sampling."""
        samples = []
        for proposal in self.proposals:
            n_samples = max(1, int(proposal['weight'] * self.n_samples_per_iter))
            accepted_samples = []

            while len(accepted_samples) < n_samples:
                # Sanitize std values
                proposal['std'] = np.nan_to_num(proposal['std'], nan=1e-6, posinf=1e6, neginf=1e-6)
                proposal['std'] = np.clip(proposal['std'], 1e-6, None)

                # Generate the covariance matrix
                cov_matrix = np.diag(proposal['std'] ** 2)

                # Debugging outputs
                #print(f"Proposal std: {proposal['std']}")
                #print(f"Covariance matrix:\n{cov_matrix}")
                #print(f"Eigenvalues: {np.linalg.eigvals(cov_matrix)}")

                # Regularize the covariance matrix
                if not np.all(np.linalg.eigvals(cov_matrix) > 0):
                    print("Regularizing covariance matrix...")
                    jitter = 1e-6
                    cov_matrix += jitter * np.eye(cov_matrix.shape[0])

                # Check for NaNs in covariance matrix
                if not np.all(np.isfinite(cov_matrix)):
                    raise ValueError("Covariance matrix contains NaN or inf values.")

                # Generate candidate samples using SciPy
                try:
                    candidate_samples = multivariate_normal.rvs(
                        mean=proposal['mean'],
                        cov=cov_matrix,
                        size=n_samples
                    )
                except np.linalg.LinAlgError as e:
                    print("Covariance matrix issue:", e)
                    break

                # Apply the condition
                filtered_samples = [sample for sample in candidate_samples if self.condition(sample)]
                accepted_samples.extend(filtered_samples[:n_samples - len(accepted_samples)])
            
            samples.extend(accepted_samples)
        return np.array(samples)

    def regularize_cov_matrix(self, cov_matrix, epsilon=1e-6):
        """
        Regularize a covariance matrix to ensure it is symmetric positive definite.
        """
        cov_matrix = np.atleast_2d(cov_matrix)  # Ensure 2D array
        return cov_matrix + np.eye(cov_matrix.shape[0]) * epsilon

    def compute_weights(self, samples):
        """Compute importance weights for the given samples."""
        weights = []
        for sample in samples:
            try:
                target_density = self.target_dist(sample)
                proposal_density = sum(
                    p['weight'] * multivariate_normal.pdf(
                        sample,
                        mean=p['mean'],
                        cov=self.regularize_cov_matrix(np.diag(np.atleast_1d(p['std']) ** 2)),
                    )
                    for p in self.proposals
                )
                weights.append(target_density / proposal_density)
            except np.linalg.LinAlgError as e:
                print(f"Error in covariance matrix: {e}")
                weights.append(0)  # Assign zero weight if an error occurs

        weights = np.array(weights)
        if np.sum(weights) == 0:
            raise ValueError("Sum of weights is zero, cannot normalize.")
        weights /= np.sum(weights)  # Normalize weights
        return weights

    def update_proposal(self, samples, weights):
        """Add a new proposal component based on weighted samples."""
        # Flatten weights
        weights = weights.flatten()
        if not np.all(np.isfinite(weights)) or np.sum(weights) == 0:
            print("Invalid or zero-sum weights. Skipping proposal update.")
            return
        if np.allclose(samples, samples[0]):
            print("All samples are identical. Skipping proposal update.")
            return

        # Sanitize weights
        weights = np.nan_to_num(weights, nan=0.0, posinf=0.0, neginf=0.0)
        weights /= np.sum(weights)  # Normalize weights

        # Sanitize samples
        samples = np.nan_to_num(samples, nan=0.0, posinf=0.0, neginf=0.0)

        # Compute the weighted mean and standard deviation for each feature
        mean = np.average(samples, axis=0, weights=weights)
        std = np.sqrt(np.average((samples - mean) ** 2, axis=0, weights=weights))

        # Add the new proposal
        new_proposal = {'mean': mean, 'std': std, 'weight': 1.0 / (len(self.proposals) + 1)}
        self.proposals.append(new_proposal)

        # Normalize weights for the mixture components
        for p in self.proposals:
            p['weight'] = 1.0 / len(self.proposals)

    def run(self):
        """Execute the adaptive sampling process."""
        for iteration in range(self.n_iterations):
            # Step 1: Sample from the current proposal
            current_samples = self.sample_from_proposal()

            # Step 2: Compute weights
            weights = self.compute_weights(current_samples)

            # Step 3: Update the proposal distribution
            self.update_proposal(current_samples, weights)

            # Step 4: Store samples and weights
            self.all_samples.extend(current_samples)
            self.all_weights.extend(weights)

            print(f"Iteration {iteration + 1}/{self.n_iterations}: New proposal added.")

        return self.all_samples, self.all_weights, self.proposals

In [3]:
class AdaptiveSampling:
    def __init__(self, target_dist, initial_proposal, n_iterations, n_samples_per_iter, condition):
        """
        Initialize the AdaptiveSampling object.

        Parameters:
        - target_dist: function, computes the target distribution density.
        - initial_proposal: list of dicts, parameters for the initial proposal distribution:
            [{'mean': ndarray, 'std': ndarray, 'weight': float}, ...].
        - n_iterations: int, number of adaptation iterations.
        - n_samples_per_iter: int, number of samples to draw per iteration.
        - condition: function, evaluates whether a sample satisfies the constraint.
        """
        self.target_dist = target_dist
        self.proposals = initial_proposal
        self.n_iterations = n_iterations
        self.n_samples_per_iter = n_samples_per_iter
        self.all_samples = []
        self.all_weights = []
        self.condition = condition

    def sample_from_proposal(self):
        """Generate samples from the current proposal mixture with conditioned sampling."""
        samples = []
        for proposal in self.proposals:
            n_samples = max(1, int(proposal['weight'] * self.n_samples_per_iter))
            accepted_samples = []

            while len(accepted_samples) < n_samples:
                # Sanitize std values
                proposal['std'] = np.nan_to_num(proposal['std'], nan=1e-6, posinf=1e6, neginf=1e-6)
                proposal['std'] = np.clip(proposal['std'], 1e-6, None)

                # Generate the covariance matrix
                cov_matrix = np.diag(proposal['std'] ** 2)

                # Debugging outputs
                #print(f"Proposal std: {proposal['std']}")
                #print(f"Covariance matrix:\n{cov_matrix}")
                #print(f"Eigenvalues: {np.linalg.eigvals(cov_matrix)}")

                # Regularize the covariance matrix
                if not np.all(np.linalg.eigvals(cov_matrix) > 0):
                    print("Regularizing covariance matrix...")
                    jitter = 1e-6
                    cov_matrix += jitter * np.eye(cov_matrix.shape[0])

                # Check for NaNs in covariance matrix
                if not np.all(np.isfinite(cov_matrix)):
                    raise ValueError("Covariance matrix contains NaN or inf values.")

                # Generate candidate samples using SciPy
                try:
                    candidate_samples = multivariate_normal.rvs(
                        mean=proposal['mean'],
                        cov=cov_matrix,
                        size=n_samples
                    )
                except np.linalg.LinAlgError as e:
                    print("Covariance matrix issue:", e)
                    break

                # Apply the condition
                filtered_samples = [sample for sample in candidate_samples if self.condition(sample)]
                accepted_samples.extend(filtered_samples[:n_samples - len(accepted_samples)])
            
            samples.extend(accepted_samples)
        return np.array(samples)

    def regularize_cov_matrix(self, cov_matrix, epsilon=1e-6):
        """
        Regularize a covariance matrix to ensure it is symmetric positive definite.
        """
        cov_matrix = np.atleast_2d(cov_matrix)  # Ensure 2D array
        return cov_matrix + np.eye(cov_matrix.shape[0]) * epsilon

    def compute_weights(self, samples):
        """Compute importance weights for the given samples."""
        weights = []
        for sample in samples:
            try:
                target_density = self.target_dist(sample)
                proposal_density = sum(
                    p['weight'] * multivariate_normal.pdf(
                        sample,
                        mean=p['mean'],
                        cov=self.regularize_cov_matrix(np.diag(np.atleast_1d(p['std']) ** 2)),
                    )
                    for p in self.proposals
                )
                weights.append(target_density / proposal_density)
            except np.linalg.LinAlgError as e:
                print(f"Error in covariance matrix: {e}")
                weights.append(0)  # Assign zero weight if an error occurs

        weights = np.array(weights)
        if np.sum(weights) == 0:
            raise ValueError("Sum of weights is zero, cannot normalize.")
        weights /= np.sum(weights)  # Normalize weights
        return weights

    def update_proposal(self, samples, weights):
        """Add a new proposal component based on weighted samples."""
        # Flatten weights
        weights = weights.flatten()
        if not np.all(np.isfinite(weights)) or np.sum(weights) == 0:
            print("Invalid or zero-sum weights. Skipping proposal update.")
            return
        if np.allclose(samples, samples[0]):
            print("All samples are identical. Skipping proposal update.")
            return

        # Sanitize weights
        weights = np.nan_to_num(weights, nan=0.0, posinf=0.0, neginf=0.0)
        weights /= np.sum(weights)  # Normalize weights

        # Sanitize samples
        samples = np.nan_to_num(samples, nan=0.0, posinf=0.0, neginf=0.0)

        # Compute the weighted mean and standard deviation for each feature
        mean = np.average(samples, axis=0, weights=weights)
        std = np.sqrt(np.average((samples - mean) ** 2, axis=0, weights=weights))

        # Add the new proposal
        new_proposal = {'mean': mean, 'std': std, 'weight': 1.0 / (len(self.proposals) + 1)}
        self.proposals.append(new_proposal)

        # Normalize weights for the mixture components
        for p in self.proposals:
            p['weight'] = 1.0 / len(self.proposals)

    def run(self):
        """Execute the adaptive sampling process."""
        for iteration in range(self.n_iterations):
            # Step 1: Sample from the current proposal
            current_samples = self.sample_from_proposal()

            # Step 2: Compute weights
            weights = self.compute_weights(current_samples)

            # Step 3: Update the proposal distribution
            self.update_proposal(current_samples, weights)

            # Step 4: Store samples and weights
            self.all_samples.extend(current_samples)
            self.all_weights.extend(weights)

            print(f"Iteration {iteration + 1}/{self.n_iterations}: New proposal added.")

        return self.all_samples, self.all_weights, self.proposals

In [2]:
def condition(sample):
        if int(round(sample[5]))!=10 and int(round(sample[5]))!=11:
             return 0
        x = sample[0:5]
        return plotting.parameter_constraints(x)

In [4]:
def initialize_multidimensional_proposal(data, n_clusters=3):
    """
    Initialize a multidimensional Gaussian proposal distribution using KMeans.
    Parameters:
    - data: ndarray, shape (n_samples, n_features), data to initialize proposals from.
    - n_clusters: int, number of clusters for initial proposals.
    Returns:
    - proposals: list of dicts, each with 'mean', 'std', and 'weight' for a Gaussian proposal.
    """
    kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(data)
    proposals = []
    for i in range(n_clusters):
        cluster_data = data[kmeans.labels_ == i]
        mean = np.mean(cluster_data, axis=0)
        std = np.std(cluster_data, axis=0)
        weight = len(cluster_data) / len(data)
        proposals.append({'mean': mean, 'std': std, 'weight': weight})
    return proposals





In [5]:
from sklearn.neighbors import KernelDensity

def estimate_target_distribution(data, bandwidth=1.0):
    """
    Estimate the target distribution using KDE for multidimensional data.
    Parameters:
    - data: ndarray, shape (n_samples, n_features), input data samples.
    - bandwidth: float, bandwidth for KDE.
    Returns:
    - target_distribution: function, computes the density of the target distribution.
    """
    kde = KernelDensity(kernel='gaussian', bandwidth=bandwidth).fit(data)
    
    def target_distribution(x):
        # Ensure x is 2D for compatibility with sklearn KDE
        x = np.atleast_2d(x)
        return np.exp(kde.score_samples(x))
    
    return target_distribution

In [ ]:
path = f'/Users/aschuetz/Documents/Analysis/legend/ML/legend-multi-fidelity-surrogate-model/simulation/out/HF/v1.2/tier5/neutron-sim-HF'
    
file_list = utils.get_all_files(path,".csv")[:-1]
# Initialize an empty DataFrame
data_train = pd.DataFrame()
# Process each file
for file_in in tqdm(file_list):
    df = pd.read_csv(file_in)
    # Append to the main DataFrame
    data_train = pd.concat([data_train, df], ignore_index=True)
    

100%|██████████| 99/99 [01:34<00:00,  1.04it/s]


In [8]:

# Set parameter name/x_labels -> needs to be consistent with data input file
#x_labels=["radius","thickness","npanels","theta","length","x_0[m]","y_0[m]","z_0[m]","px_0[m]","py_0[m]","pz_0[m]","ekin_0[eV]"]
x_labels=["radius","thickness","npanels","theta","length","muon_type","muon_ekin","muon_xloc","muon_yloc","muon_zloc","muon_theta","muon_phi"]
y_label = 'nC_Ge77'


x_lf = data_train[x_labels].to_numpy()
y_lf = data_train[y_label].to_numpy()
x_lf_sig = data_train[(data_train[y_label] >= 1)][x_labels].to_numpy()

# Output the result
print(f"Total rows with y(x) = 1: {x_lf_sig.shape[0]}")

Total rows with y(x) = 1: 389


In [9]:
target_distribution = estimate_target_distribution(x_lf)

In [10]:
# Initialize Gaussian proposals from 5D data
ini_proposals = initialize_multidimensional_proposal(x_lf_sig, n_clusters=2)

/Users/aschuetz/.local/modules/miniconda/miniconda3/envs/legend/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [11]:
# Output the proposals
print("Gaussian Proposals:")
for i, proposal in enumerate(ini_proposals):
    print(f"Cluster {i + 1}:")
    print(f"  Mean: {proposal['mean']}")
    print(f"  Std:  {proposal['std']}")
    print(f"  Weight: {proposal['weight']:.2f}")

Gaussian Proposals:
Cluster 1:
  Mean: [ 1.91803125e+02  1.06906250e+01  9.40000000e+01  3.13906250e+01
  1.86718750e+01  1.05312500e+01  3.81345313e+03  9.59406250e+01
 -2.78228125e+02  5.33362500e+02  7.55256250e-01  2.66337187e+00]
  Std:  [4.28761478e+01 3.97400612e+00 7.76313403e+01 2.61707656e+01
 2.43507473e+01 4.99022482e-01 1.81275763e+03 5.53360405e+02
 4.77892937e+02 3.70362123e+02 3.05218551e-01 1.63742379e+00]
  Weight: 0.08
Cluster 2:
  Mean: [ 1.88850420e+02  1.04375350e+01  1.28613445e+02  3.69145658e+01
  1.15929972e+01  1.04005602e+01  5.20128291e+02  1.80319328e+01
 -1.62353221e+02  6.89926050e+02  6.72292157e-01  2.67126807e+00]
  Std:  [4.54826883e+01 3.60548020e+00 9.42823499e+01 2.67245208e+01
 1.75569815e+01 4.90011970e-01 5.31918150e+02 5.89867871e+02
 4.92976371e+02 3.85567586e+02 2.73212899e-01 1.81570230e+00]
  Weight: 0.92


In [15]:


# Initialize the AdaptiveSampling class
adaptive_sampler = AdaptiveSampling(
    target_dist=target_distribution,
    initial_proposal=ini_proposals,
    n_iterations=5,
    n_samples_per_iter=250,
    condition=condition
)

In [16]:
# Run the adaptive sampling process
samples, weights, proposals = adaptive_sampler.run()


Iteration 1/5: New proposal added.
Iteration 2/5: New proposal added.
Iteration 3/5: New proposal added.
Iteration 4/5: New proposal added.
Iteration 5/5: New proposal added.


In [ ]:
print("Final Proposals:")
for i, proposal in enumerate(adaptive_sampler.proposals):
    mean = ", ".join([f"{v:.2f}" for v in proposal['mean']])
    std = ", ".join([f"{v:.2f}" for v in proposal['std']])
    print(f"Component {i + 1}: Mean=[{mean}], Std=[{std}], Weight={proposal['weight']:.4f}")

In [ ]:
len(samples)

In [ ]:

for i,x in enumerate(samples):
    sim.print_geant4_macro_adaptive(x[0:5],f"{i}",mode="HF",version="v1.4")
    sim.print_musun([x[4:]],"./data/musun/",i)

In [ ]:
t=5.
print(round(t))
if int(round(t))!=10 and int(round(t))!=11:
    print(0)